# Day19 — Isolated Unity Worker

## Goal

This offline tutorial demonstrates the immutable job/result contract, ordered compile → EditMode → PlayMode gates, cancellation and integrity rejection. It uses deterministic fake evidence: it does not prove a real Unity Editor run, enforced OS network isolation, or a real remote HTTPS deployment.

## Contract

In [ ]:
from datetime import datetime, timedelta, timezone
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()).lower() == 'day19' else os.getcwd()
if project_root not in sys.path: sys.path.insert(0, project_root)
from tools.unity_worker_contract import build_job_manifest, build_worker_result, validate_job_manifest, validate_worker_result
from ui.view_state import worker_validation_view
now = datetime(2026, 8, 21, 8, tzinfo=timezone.utc)
digest_a, digest_b = 'a' * 64, 'b' * 64
def job_for(gate, attempt=1):
    return build_job_manifest(thread_id='day19-demo', attempt=attempt, gate=gate, snapshot_sha256=digest_a, unity_version='2022.3.62f2c1', package_manifest_sha256=digest_b, timeout_seconds=60, expires_at=(now + timedelta(minutes=10)).isoformat().replace('+00:00', 'Z'), network_policy={'mode': 'disabled', 'allowlist': []}, files=[{'path': 'Assets/Generated/Probe.cs', 'size': 24, 'sha256': digest_a}])
jobs = [job_for(gate, index) for index, gate in enumerate(('compile', 'editmode', 'playmode'), 1)]
assert all(validate_job_manifest(job) == [] for job in jobs)
print([job['gate'] for job in jobs])

## Snapshot

A production run packages only allowlisted Unity inputs and binds the archive hash, Unity version, package manifest hash, file hashes, timeout, and network policy into the immutable job identity. The digest values below are offline fixtures, not a real project archive.

In [ ]:
assert jobs[0]['snapshot_sha256'] == digest_a
assert jobs[0]['network_policy'] == {'mode': 'disabled', 'allowlist': []}
assert jobs[0]['job_id'] != jobs[1]['job_id']
print('snapshot-bound identities verified')

## Local Fake Worker

The fake worker returns schema-valid terminal evidence without launching Unity or opening a network listener.

In [ ]:
def passed_result(job, total):
    return build_worker_result(job, status='passed', worker_id='offline-worker', started_at=now.isoformat().replace('+00:00', 'Z'), finished_at=(now + timedelta(seconds=2)).isoformat().replace('+00:00', 'Z'), failure_owner='', error_code='', evidence={'compiler_errors': [], 'test_summary': {'total': total, 'passed': total, 'failed': 0, 'skipped': 0, 'inconclusive': 0, 'duration': 0.2}}, artifacts=[], cleanup={'sandbox_removed': True, 'process_stopped': True})
results = [passed_result(job, 0 if job['gate'] == 'compile' else 2) for job in jobs]
assert all(validate_worker_result(job, result, now=now) == [] for job, result in zip(jobs, results))
print([(result['gate'], result['status']) for result in results])

## EditMode/PlayMode Results

EditMode and PlayMode remain independent authoritative gates; both must pass before review and local Git commit.

In [ ]:
state = {'unity_worker_mode': 'local', 'unity_worker_jobs': [{**jobs[-1], **{'status': 'passed', 'worker_id': 'offline-worker', 'started_at': results[-1]['started_at'], 'finished_at': results[-1]['finished_at']}}], 'editmode_test_result': {'success': True, 'worker_status': 'passed', 'summary': results[1]['evidence']['test_summary']}, 'playmode_test_result': {'success': True, 'worker_status': 'passed', 'summary': results[2]['evidence']['test_summary']}}
safe_view = worker_validation_view(state, now=now)
assert safe_view['editmode']['total'] == safe_view['playmode']['total'] == 2
print(safe_view)

## Cancel/Timeout

In [ ]:
empty_summary = {'total': 0, 'passed': 0, 'failed': 0, 'skipped': 0, 'inconclusive': 0, 'duration': 0.0}
cancelled = build_worker_result(jobs[1], status='cancelled', worker_id='offline-worker', started_at=results[1]['started_at'], finished_at=results[1]['finished_at'], failure_owner='worker', error_code='WORKER_CANCELLED', evidence={'compiler_errors': [], 'test_summary': empty_summary}, artifacts=[], cleanup={'sandbox_removed': True, 'process_stopped': True})
timed_out = build_worker_result(jobs[2], status='timed_out', worker_id='offline-worker', started_at=results[2]['started_at'], finished_at=results[2]['finished_at'], failure_owner='timeout', error_code='WORKER_TIMEOUT', evidence={'compiler_errors': [], 'test_summary': empty_summary}, artifacts=[], cleanup={'sandbox_removed': True, 'process_stopped': True})
assert validate_worker_result(jobs[1], cancelled, now=now) == []
assert validate_worker_result(jobs[2], timed_out, now=now) == []
print(cancelled['status'], timed_out['status'])

## Stale Result

In [ ]:
stale_errors = validate_worker_result(jobs[0], results[0], now=now + timedelta(hours=1))
assert 'job result is expired' in stale_errors
print(stale_errors)

## Security

The controller submits only a fixed manifest and archive. The worker has fixed routes and arguments, rejects unknown fields and stale results, validates hashes, uses bounded artifacts, and exposes no arbitrary command endpoint. Non-loopback remote mode requires HTTPS plus signed timestamp/nonce requests. Network-disabled claims require independently enforced isolation.

In [ ]:
tampered = dict(jobs[0]); tampered['command'] = 'arbitrary'
assert 'unknown job fields: command' in validate_job_manifest(tampered)
assert set(safe_view) == {'mode', 'worker_id', 'gate', 'status', 'elapsed_seconds', 'error_code', 'editmode', 'playmode'}
print('fail-closed checks verified')

## Next Steps

Run the full offline suite, then record real local Unity 2022.3 evidence and real remote HTTPS/network-isolation evidence separately. Keep every unexecuted acceptance probe marked PENDING.